In [1]:
import json
import os
import re
import csv
import pandas as pd
from tqdm import tqdm
import gzip

In [2]:
def process_scrutin(scrutin: dict = None):

    all_votants = []

    main_data_keys = [
        "uid",
        "numero",
        "organeRef",
        "legislature",
        "seanceRef",
        "dateScrutin",
        "quantiemeJourSeance",
        "typeVote/codeTypeVote",
        "typeVote/typeMajorité",
        "sort/code",
        "titre",
        "objet/libelle",
        "syntheseVote/nombreVotants",
        "syntheseVote/suffragesExprimes",
        "syntheseVote/nbrSuffragesRequis",
        "syntheseVote/annonce",
        "syntheseVote/decompte/nonVotants",
        "syntheseVote/decompte/pour",
        "syntheseVote/decompte/contre",
        "syntheseVote/decompte/abstentions",
        "syntheseVote/decompte/nonVotantsVolontaires",
    ]

    main_data = {}

    for item in main_data_keys:
        parts = item.split("/")
        init_part = scrutin.get(parts[0])
        for key in parts[1:]:
            init_part = init_part.get(key)
        main_data[f"main/{item}"] = init_part

    for _, institution in scrutin.get("ventilationVotes").items():
        institution_organeRef = institution.get("organeRef")

        for group in institution.get("groupes").get("groupe"):
            data_group_keys = [
                "organeRef",
                "nombreMembresGroupe",
                "vote/positionMajoritaire",
                "vote/decompteVoix/nonVotants",
                "vote/decompteVoix/pour",
                "vote/decompteVoix/contre",
                "vote/decompteVoix/abstentions",
                "vote/decompteVoix/nonVotantsVolontaires"
            ]

            group_data = {}

            for item in data_group_keys:
                parts = item.split("/")
                init_part = group.get(parts[0])
                for key in parts[1:]:
                    init_part = init_part.get(key)
                group_data[f"groupe/{item}"] = init_part



            for votant_type, votant_dict in group.get("vote").get("decompteNominatif").items():
                if votant_dict is None:
                    continue

                if votant_dict == "0":
                    continue

                # print(votant_type)

                votant_list = votant_dict.get("votant")

                if type(votant_list) == dict:
                    votant_payload = {}
                    votant_payload.update({f"votant/{k}": str(v) for k, v in votant_list.items()})
                    votant_payload.update({k: str(v) for k, v in group_data.items()})
                    votant_payload.update({k: str(v) for k, v in main_data.items()})
                    votant_payload.update({"votant/vote": votant_type})
                    all_votants.append(votant_payload)
                    continue

                for votant in votant_list:
                    votant_payload = {}
                    votant_payload.update({f"votant/{k}": str(v) for k, v in votant.items()})
                    votant_payload.update({k: str(v) for k, v in group_data.items()})
                    votant_payload.update({k: str(v) for k, v in main_data.items()})
                    votant_payload.update({"votant/vote": votant_type})
                    all_votants.append(votant_payload)

    return all_votants

In [3]:
# Cell: Unzip Scrutins_XV.json.zip
import zipfile

# Define paths
here = os.getcwd()
folder = os.path.dirname(here)
folder_0 = folder + "/0_raw"
zip_path = os.path.join(folder_0, "Scrutins.json.zip") 
extract_path = os.path.join(folder_0, "json")

# Create extraction directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip the file
print(f"Unzipping {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(folder_0)  # ← Extract to folder_0, not extract_path
    
print(f"Extracted to: {folder_0}")

# Check what was extracted
json_dir = os.path.join(folder_0, "json")
if os.path.exists(json_dir):
    json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
    print(f"Found {len(json_files)} JSON files in: {json_dir}")
else:
    print(f"ERROR: 'json' folder not found in {folder_0}")

Unzipping c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\16e_legislature_ 28juin2022-9juin2024/0_raw\Scrutins.json.zip...
Extracted to: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\16e_legislature_ 28juin2022-9juin2024/0_raw
Found 4106 JSON files in: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\16e_legislature_ 28juin2022-9juin2024/0_raw\json


In [4]:
# Cell: Load and process JSON files
database = []

# Point to the extracted JSON directory
data_dir = os.path.join(folder_0, "json")

for filename in tqdm(os.listdir(data_dir)):
    if not re.match("^.*\.json$", filename):
        continue

    source_filename = os.path.join(data_dir, filename)

    with open(source_filename, "r", encoding="UTF-8") as input_file:
        content = json.load(input_file)

    all_votants = process_scrutin(scrutin=content.get("scrutin"))
    database.extend(all_votants)

df = pd.DataFrame.from_records(database)

# Save to CSV only
output_path = os.path.join(folder, "1_extract_python/scrutins.csv")
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Dataset shape: {df.shape}")

  0%|          | 0/4106 [00:00<?, ?it/s]

100%|██████████| 4106/4106 [01:36<00:00, 42.74it/s]


Saved to: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\16e_legislature_ 28juin2022-9juin2024\1_extract_python/scrutins.csv
Dataset shape: (603813, 34)


In [8]:
df["main/dateScrutin"].sort_values().unique()

array(['2022-07-11', '2022-07-12', '2022-07-13', '2022-07-18',
       '2022-07-19', '2022-07-20', '2022-07-21', '2022-07-22',
       '2022-07-23', '2022-07-25', '2022-07-26', '2022-07-27',
       '2022-07-28', '2022-08-02', '2022-08-03', '2022-08-04',
       '2022-10-03', '2022-10-04', '2022-10-05', '2022-10-06',
       '2022-10-10', '2022-10-11', '2022-10-12', '2022-10-13',
       '2022-10-14', '2022-10-17', '2022-10-18', '2022-10-19',
       '2022-10-20', '2022-10-24', '2022-10-25', '2022-10-26',
       '2022-10-27', '2022-10-28', '2022-10-31', '2022-11-04',
       '2022-11-07', '2022-11-08', '2022-11-14', '2022-11-15',
       '2022-11-16', '2022-11-17', '2022-11-18', '2022-11-21',
       '2022-11-22', '2022-11-23', '2022-11-24', '2022-11-25',
       '2022-11-28', '2022-11-29', '2022-11-30', '2022-12-01',
       '2022-12-02', '2022-12-05', '2022-12-06', '2022-12-07',
       '2022-12-08', '2022-12-09', '2022-12-11', '2022-12-12',
       '2022-12-13', '2022-12-14', '2022-12-15', '2022-

In [1]:
import pandas as pd 
import os
here  = os.getcwd()
df = pd.read_csv(os.path.join(here, "scrutins.csv"))
df.columns.tolist()

['votant/acteurRef',
 'votant/mandatRef',
 'votant/causePositionVote',
 'groupe/organeRef',
 'groupe/nombreMembresGroupe',
 'groupe/vote/positionMajoritaire',
 'groupe/vote/decompteVoix/nonVotants',
 'groupe/vote/decompteVoix/pour',
 'groupe/vote/decompteVoix/contre',
 'groupe/vote/decompteVoix/abstentions',
 'groupe/vote/decompteVoix/nonVotantsVolontaires',
 'main/uid',
 'main/numero',
 'main/organeRef',
 'main/legislature',
 'main/seanceRef',
 'main/dateScrutin',
 'main/quantiemeJourSeance',
 'main/typeVote/codeTypeVote',
 'main/typeVote/typeMajorité',
 'main/sort/code',
 'main/titre',
 'main/objet/libelle',
 'main/syntheseVote/nombreVotants',
 'main/syntheseVote/suffragesExprimes',
 'main/syntheseVote/nbrSuffragesRequis',
 'main/syntheseVote/annonce',
 'main/syntheseVote/decompte/nonVotants',
 'main/syntheseVote/decompte/pour',
 'main/syntheseVote/decompte/contre',
 'main/syntheseVote/decompte/abstentions',
 'main/syntheseVote/decompte/nonVotantsVolontaires',
 'votant/vote',
 'votan